# Python Intern Technical Test

## What this is
A progressive test: Python basics -> data structures -> OOP & dataclasses -> data manipulation (pandas) -> visualization -> a small FastAPI backend (+ optional front-end) -> a short system-design question.

## Rules - please read
1. **Use AI freely** (ChatGPT, Claude, Copilot, anything). We use it every day and we expect you to.
2. **But you must be able to explain every line you submit.** After you submit there is a **45-60 min call**: you will walk us through your work, answer "why" questions, predict what happens when we change something, and make a **small live change** to your own code. Code you cannot explain counts against you more than code that is missing.
3. Keep a short log in `AI_LOG.md` (template in the repo): which tool you used for what, one thing the AI got wrong or that you rejected, and one thing you learned.
4. Written answers (the "Your answer" cells) can be in **English or Portuguese**. Keep them short: 2-5 sentences. Your own words, please.
5. If you get stuck, **skip and move on**, and tell us in the notebook what you tried. A partial but understood solution beats a complete mystery.

## Parts
| Part | Topic |
|---|---|
| 1 | Python basics & data structures |
| 2 | Classes, dataclasses, OOP |
| 3 | Data manipulation with pandas |
| 4 | Visualization |
| 5 | FastAPI backend (`app/`) |
| 6 | System design (`DESIGN_ANSWER.md`) |
| 7 | AI log, clean-up, commit |

## How to submit
Fork/clone this repo -> work in a branch -> *Restart kernel & Run All* so the notebook is saved with outputs -> make sure `pytest` runs -> push and send us the link.

In [ ]:
import sys, re, math, dataclasses
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict, replace
from enum import Enum
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT))          # so `import app` works from the notebook


def check(label, fn):
    """Self-check helper: prints OK / FAIL and never stops the notebook."""
    try:
        ok = bool(fn())
    except Exception as e:             # a check can fail because your code raised
        print(f"[FAIL] {label}   ({type(e).__name__}: {e})")
        return False
    print(("[ OK ] " if ok else "[FAIL] ") + label)
    return ok


def raises(exc, fn):
    """True if fn() raises `exc`."""
    try:
        fn()
    except exc:
        return True
    except Exception:
        return False
    return False


print("root:", ROOT, "| data files:", sorted(p.name for p in DATA.glob("*")))

---
# Part 1 - Python basics & data structures

A 96-well plate has 8 rows (`A`-`H`) and 12 columns (`1`-`12`). Our lab software numbers wells **column by column**:
`A1, B1, ... H1, A2, B2, ... H12`  (index 0 -> `A1`, index 7 -> `H1`, index 8 -> `A2`, index 95 -> `H12`).

### 1.1 Well names
Implement both functions. They must raise `ValueError` on invalid input, and `well_index` must tolerate letter case and surrounding spaces.

In [ ]:
ROWS = "ABCDEFGH"
N_COLS = 12


def well_name(index: int) -> str:
    """0 -> 'A1', 7 -> 'H1', 8 -> 'A2', 95 -> 'H12'. ValueError if index is outside 0..95."""
    raise NotImplementedError


def well_index(name: str) -> int:
    """Inverse of well_name. ' b3 ' -> 17. ValueError for 'I1', 'A13', 'A0', '', 'A01'..."""
    raise NotImplementedError

In [ ]:
check("well_name(0) == 'A1'", lambda: well_name(0) == "A1")
check("well_name(8) == 'A2'", lambda: well_name(8) == "A2")
check("well_name(95) == 'H12'", lambda: well_name(95) == "H12")
check("well_name(96) raises", lambda: raises(ValueError, lambda: well_name(96)))
check("well_name(-1) raises", lambda: raises(ValueError, lambda: well_name(-1)))
check("well_index(' b3 ') == 17", lambda: well_index(" b3 ") == 17)
check("round trip for all 96 wells", lambda: all(well_index(well_name(i)) == i for i in range(96)))
check("invalid wells raise", lambda: all(raises(ValueError, lambda w=w: well_index(w)) for w in ["I1", "A13", "A0", "", "A01", "11"]))

### 1.2 Parsing a messy patient list
`data/patients_raw.txt` is a list exported by another system, one `sample_id, well` per line. Look at the file first - it has comments, blank lines, odd spacing/casing and some bad lines on purpose.

Write `parse_patient_lines(lines)` returning `(valid, errors)`:
* `valid`: list of `(sample_id, well)` - sample id exactly as written (just stripped), well normalised (`' e3'` -> `'E3'`).
* `errors`: list of `(line_number, stripped_line, reason)` - `line_number` is 1-based **in the original file** (comments and blanks count).
* Blank lines and lines starting with `#` are ignored silently.
* A line is an **error** (and skipped) if: it doesn't have exactly 2 comma-separated fields, the sample id is empty, the well is invalid, or the well was already used by an earlier valid line.

In [ ]:
def parse_patient_lines(lines: Iterable[str]):
    """Return (valid, errors) as described above."""
    raise NotImplementedError

In [ ]:
lines = (DATA / "patients_raw.txt").read_text(encoding="utf-8").splitlines()
valid, errors = parse_patient_lines(lines)
print(len(valid), "valid;", len(errors), "errors")
for e in errors:
    print("  line", e[0], "->", e[2])

check("7 valid lines", lambda: len(valid) == 7)
check("pat_003 keeps its case, well normalised", lambda: ("pat_003", "E3") in valid)
check("errors point at the right original lines", lambda: [e[0] for e in errors] == [8, 9, 10, 11, 12, 13])
check("no duplicate wells in valid", lambda: len({w for _, w in valid}) == len(valid))

### 1.3 Data structures
Write these three small functions (no libraries needed):

In [ ]:
def group_by_column(wells: list) -> dict:
    """['A1','B1','A2'] -> {1: ['A1','B1'], 2: ['A2']}  (keep input order inside each list)."""
    raise NotImplementedError


def find_duplicates(ids: list) -> set:
    """Return the ids that appear more than once."""
    raise NotImplementedError


def invert(mapping: dict) -> dict:
    """{'PAT1': 'A1', 'PAT2': 'A1', 'PAT3': 'B1'} -> {'A1': ['PAT1', 'PAT2'], 'B1': ['PAT3']}"""
    raise NotImplementedError

In [ ]:
check("group_by_column", lambda: group_by_column(["A1", "B1", "A2"]) == {1: ["A1", "B1"], 2: ["A2"]})
check("group_by_column empty", lambda: group_by_column([]) == {})
check("find_duplicates", lambda: find_duplicates(["a", "b", "a", "c", "c", "c"]) == {"a", "c"})
check("find_duplicates none", lambda: find_duplicates(["a", "b"]) == set())
check("invert", lambda: invert({"PAT1": "A1", "PAT2": "A1", "PAT3": "B1"}) == {"A1": ["PAT1", "PAT2"], "B1": ["PAT3"]})

**Your answer - lists, tuples, sets, dicts.** For each of the four, say *when you would pick it* and give **one concrete example from the plate/patient domain** (e.g. "I'd use a ___ to store ___ because ___"). 2 sentences each.

> **list:** _(write here)_
>
> **tuple:** _(write here)_
>
> **set:** _(write here)_
>
> **dict:** _(write here)_

### 1.4 Find the bugs
Both functions below look fine but are wrong. For each: write a minimal call that **shows the bug**, then fix it and explain the cause in one sentence.

In [ ]:
def add_sample(sample_id, well, registry=[]):
    registry.append((sample_id, well))
    return registry


def last_n_wells(wells, n):
    """Return the last n wells (all of them if n > len(wells), none if n == 0)."""
    return wells[len(wells) - n + 1:]

In [ ]:
# show the bugs here (before fixing), then fix the functions above and re-run the checks
check("add_sample doesn't leak between calls", lambda: add_sample("a", "A1") != add_sample("b", "B1") and len(add_sample("c", "C1")) == 1)
check("last_n_wells(5 items, 2)", lambda: last_n_wells(list("ABCDE"), 2) == ["D", "E"])
check("last_n_wells n == 0", lambda: last_n_wells(list("ABCDE"), 0) == [])
check("last_n_wells n > len", lambda: last_n_wells(list("ABC"), 7) == ["A", "B", "C"])

**Your answer - bugs.** What was wrong in each function, and why did it behave that way?

> `add_sample`: _(write here)_
>
> `last_n_wells`: _(write here)_

---
# Part 2 - Classes, dataclasses & OOP

### 2.0 Warm-up: what a dataclass gives you
Read and run this. A `@dataclass` writes `__init__`, `__repr__` and `__eq__` for you from the annotated fields.

In [ ]:
@dataclass
class Reagent:
    name: str
    volume_ul: float
    lot: str = "n/a"
    notes: list = field(default_factory=list)   # a mutable default MUST use default_factory (`= []` raises ValueError)


r = Reagent("Wash buffer", 1500.0)
print(r)                                          # auto __repr__
print(r == Reagent("Wash buffer", 1500.0))        # auto __eq__ (field by field)
print(asdict(r))                                  # -> plain dict
print(replace(r, lot="L42"))                      # copy with one field changed

### 2.1 `Sample` - a (frozen) dataclass
Build `Sample` with `@dataclass(frozen=True)`:

| field | type | default |
|---|---|---|
| `sample_id` | `str` | - |
| `well` | `str` | - |
| `sample_type` | `SampleType` | `SampleType.patient` |
| `dilution` | `int` | `1` |
| `tags` | `tuple` | `()` |

Behaviour:
* `well` is **normalised on creation** (`'c2 '` -> `'C2'`) and must be a valid well (`ValueError` otherwise). Reuse `well_name/well_index` from Part 1.
* `dilution` must be `>= 1` (`ValueError` otherwise).
* property `plate_index` -> the column-major index of the well (`C2` -> `10`).
* property `is_control` -> `True` only for negative/positive controls.
* Two samples with the same field values must be equal, and be usable as dict keys / in sets.

In [ ]:
class SampleType(str, Enum):
    blank = "blank"
    standard = "standard"
    neg_control = "neg_control"
    pos_control = "pos_control"
    patient = "patient"


@dataclass(frozen=True)
class Sample:
    ...   # TODO

In [ ]:
a = Sample("X", "a1")
check("well normalised", lambda: a.well == "A1")
check("defaults", lambda: a.sample_type is SampleType.patient and a.dilution == 1 and a.tags == ())
check("equality by value", lambda: a == Sample("X", "A1"))
check("hashable / set-able", lambda: len({a, Sample("X", "A1"), Sample("Y", "A1")}) == 2)
check("frozen", lambda: raises(dataclasses.FrozenInstanceError, lambda: setattr(a, "well", "B1")))
check("replace() gives a modified copy", lambda: replace(a, well="B1").well == "B1" and a.well == "A1")
check("plate_index", lambda: Sample("X", "C2").plate_index == 10)
check("is_control", lambda: Sample("N", "A2", SampleType.neg_control).is_control and not a.is_control)
check("invalid well -> ValueError", lambda: raises(ValueError, lambda: Sample("X", "Z9")))
check("dilution 0 -> ValueError", lambda: raises(ValueError, lambda: Sample("X", "A1", dilution=0)))
check("asdict works", lambda: asdict(a)["well"] == "A1")

### 2.2 The same thing as a plain class
Now write `LegacySample`: **a normal class** (no `@dataclass`) with the same behaviour as `Sample` for: construction + well normalisation + dilution check, `__repr__`, `__eq__`, `__hash__`, and the two properties. (It does not need to be frozen.)

In [ ]:
class LegacySample:
    ...   # TODO

In [ ]:
b = LegacySample("X", " a1 ")
check("well normalised", lambda: b.well == "A1")
check("equality by value", lambda: b == LegacySample("X", "A1"))
check("different objects with same data hash equally", lambda: hash(b) == hash(LegacySample("X", "A1")))
check("not equal to other data", lambda: b != LegacySample("Y", "A1"))
check("repr shows the values", lambda: "sample_id='X'" in repr(b) and "'A1'" in repr(b))
check("plate_index / is_control", lambda: LegacySample("N", "C2", SampleType.neg_control).plate_index == 10 and LegacySample("N", "C2", SampleType.neg_control).is_control)
check("dilution 0 -> ValueError", lambda: raises(ValueError, lambda: LegacySample("X", "A1", dilution=0)))

**Your answer - dataclass vs class.**

> 1. List what `@dataclass` generated for you that you had to write by hand in `LegacySample`: _(write here)_
>
> 2. In `Sample.__post_init__`, why can't you simply write `self.well = ...`? What did you do instead (or what would you do)? _(write here)_
>
> 3. Give one situation *in a project like ours* where you'd still choose a plain class over a dataclass: _(write here)_

### 2.3 `Plate` - a class that holds samples
Build `Plate`, which stores `Sample`s by well. Requirements:

* **class attributes** `ROWS` and `N_COLS` (shared by all plates)
* `__init__(self, name)` -> instance attribute `name`, and private storage of samples keyed by well
* property `n_wells` -> `96` (computed from the class attributes)
* `assign(sample)` -> store it; `ValueError` if that well is already taken
* `get(well)` -> the `Sample` in that well (well can be any case/spacing) or `None`
* `free_wells()` -> list of empty wells in plate (column-major) order
* `@classmethod from_samples(name, samples)` -> build and fill a plate
* `@staticmethod is_valid_well(name)` -> `bool` (never raises)
* dunder methods: `__len__` (number of assigned samples), `__contains__(well)`, `__iter__` (samples in **plate order**, not insertion order), `__repr__` -> `Plate('P2', 2/96 wells)`

In [ ]:
class Plate:
    ...   # TODO

In [ ]:
p = Plate("P1")
s = Sample("PAT_001", "c2")
p.assign(s)
check("n_wells == 96, len == 1", lambda: p.n_wells == 96 and len(p) == 1)
check("get() is case/space tolerant", lambda: p.get(" c2 ") == s and p.get("D2") is None)
check("'C2' in plate", lambda: "c2" in p and "D2" not in p and "ZZ" not in p)
check("assigning a taken well raises", lambda: raises(ValueError, lambda: p.assign(Sample("OTHER", "C2"))))
check("free_wells", lambda: len(p.free_wells()) == 95 and p.free_wells()[0] == "A1" and "C2" not in p.free_wells())
p2 = Plate.from_samples("P2", [Sample("a", "H12"), Sample("b", "A1")])
check("from_samples + iteration in plate order", lambda: [x.well for x in p2] == ["A1", "H12"])
check("repr", lambda: repr(p2) == "Plate('P2', 2/96 wells)")
check("is_valid_well never raises", lambda: Plate.is_valid_well("A1") and not Plate.is_valid_well("Q9") and not Plate.is_valid_well(None))
check("class attrs shared, instance attrs not", lambda: Plate.ROWS == "ABCDEFGH" and p.name != p2.name)

### 2.4 Predict, then explain
**Before running the next cell**, write what you think it prints, in the markdown cell below it. Then run it, explain what happened and fix the class (`BatchFixed`).

In [ ]:
class Batch:
    samples = []                       # <- look closely

    def __init__(self, name):
        self.name = name

    def add(self, sample_id):
        self.samples.append(sample_id)


b1, b2 = Batch("first"), Batch("second")
b1.add("PAT_001")
print(b2.samples)

> **My prediction (written before running):** _(write here)_
>
> **What actually happened and why:** _(write here)_

In [ ]:
class BatchFixed:
    ...   # TODO: same interface as Batch, no surprise

In [ ]:
f1, f2 = BatchFixed("first"), BatchFixed("second")
f1.add("PAT_001")
check("BatchFixed instances don't share samples", lambda: f1.samples == ["PAT_001"] and f2.samples == [])

### 2.5 Vocabulary, with your own code
Fill the table using **`Plate`** (from 2.3) as the source of every example - name the exact line/member and say in a few words what makes it that kind of thing.

| Concept | Example in `Plate` | What makes it that? |
|---|---|---|
| class attribute | | |
| instance attribute | | |
| method | | |
| `@property` | | |
| `@classmethod` | | |
| `@staticmethod` | | |

> _(write here)_  <- delete this line once the table above is filled in

---
# Part 3 - Data manipulation with pandas

Two files in `data/`:
* `plate_layout.csv` - what is in each of the 96 wells (`well, sample_id, sample_type, std_conc`).
* `readings_raw.csv` - what the reader exported, **long format**: one row per `(well, analyte)` with `mfi` (signal) and `bead_count` (how many beads were measured). There are 4 analytes (`AN_A` ... `AN_D`). It is **messy on purpose**.

### Calling rules  *(follow these exactly)*
1. **Clean:** well names -> stripped + upper-case; `mfi` -> number (decimal comma `"23,5"` -> `23.5`; anything unreadable such as `"OOR >"` or empty -> `NaN`); drop **exact duplicate rows**. After cleaning, `(well, analyte)` must be unique (384 rows).
2. **Join** the layout on `well`.
3. **Blank subtraction:** for each analyte, `blank_mean` = mean `mfi` of the `blank` wells (ignore NaN). `net_mfi = mfi - blank_mean`. Do **not** clip negatives.
4. A reading is **valid** if `mfi` is not NaN **and** `bead_count >= 35`.
5. **Cutoff per analyte** = `mean(net_mfi) + 3 * std(net_mfi, ddof=1)` computed over the **valid** `neg_control` wells of that analyte. If fewer than 2 valid neg-control readings exist, the cutoff is undefined.
6. **Call** per row: `INVALID` if the reading is not valid (or the cutoff is undefined), else `POS` if `net_mfi > cutoff`, else `NEG`.
7. **Run QC:** the run is valid only if, for every analyte, there is at least one valid `pos_control` reading and **all** valid `pos_control` readings are `POS`.

### 3.0 Explore

In [ ]:
raw = pd.read_csv(DATA / "readings_raw.csv")
layout = pd.read_csv(DATA / "plate_layout.csv")
display(raw.head(8))
raw.info()
display(layout.sample_type.value_counts())

**Your answer - data quality.** List every data-quality problem you found in `readings_raw.csv` (at least 5) and, next to each, how you detected it (which pandas call / what you looked at).

> _(write here)_

### 3.1 Clean

In [ ]:
def clean_readings(raw: pd.DataFrame) -> pd.DataFrame:
    """Rule 1. Must not modify `raw` in place."""
    raise NotImplementedError


clean = clean_readings(raw)

In [ ]:
check("384 rows (96 wells x 4 analytes)", lambda: len(clean) == 384)
check("(well, analyte) is unique", lambda: not clean.duplicated(["well", "analyte"]).any())
check("all wells valid", lambda: clean.well.map(Plate.is_valid_well).all())
check("mfi is numeric", lambda: pd.api.types.is_float_dtype(clean.mfi))
check("6 unusable mfi values became NaN (4 empty + 2 'OOR >')", lambda: clean.mfi.isna().sum() == 6)
check("input untouched", lambda: len(raw) == 387)

### 3.2 Join the layout

In [ ]:
df = ...   # clean + layout[["well", "sample_id", "sample_type"]] on "well"

In [ ]:
check("still 384 rows", lambda: len(df) == 384)
check("every row has a sample_type", lambda: df.sample_type.notna().all())

### 3.3 - 3.5 Blank subtraction, cutoffs, calls

In [ ]:
# 3.3 blank_mean: Series indexed by analyte. Then add the column df["net_mfi"].
blank_mean = ...
df["net_mfi"] = ...

# 3.4 valid readings + cutoffs: Series indexed by analyte
df["valid"] = ...
cutoffs = ...

# 3.5 the call per row -> add df["call"] with values "POS" / "NEG" / "INVALID"
df["call"] = ...
results = df

In [ ]:
print(cutoffs.round(1).to_dict())
print(results.call.value_counts().to_dict())
check("blank_mean has 4 analytes", lambda: len(blank_mean) == 4)
check("cutoffs match the rules (+-0.1)", lambda: all(abs(cutoffs[k] - v) < 0.1 for k, v in {'AN_A': 30.6, 'AN_B': 34.5, 'AN_C': 16.7, 'AN_D': 35.0}.items()))
check("call counts: 295 NEG / 78 POS / 11 INVALID", lambda: results.call.value_counts().to_dict() == {'NEG': 295, 'POS': 78, 'INVALID': 11})
check("blank wells are NEG or INVALID, never POS", lambda: not ((results.sample_type == "blank") & (results.call == "POS")).any())

### 3.6 Summaries

Build `summary`: **index = analyte**, columns `n_valid`, `n_pos`, `pos_rate` (%, rounded to 1 decimal), considering **patient wells with a valid reading only**. Then compute `run_valid` (rule 7) as a Python `bool`.

In [ ]:
summary = ...
run_valid = ...

In [ ]:
display(summary)
check("summary index/columns", lambda: list(summary.index) == ["AN_A", "AN_B", "AN_C", "AN_D"] and {"n_valid", "n_pos", "pos_rate"} <= set(summary.columns))
check("AN_A numbers", lambda: summary.loc["AN_A", "n_valid"] == 78 and summary.loc["AN_A", "n_pos"] == 14)
check("run_valid is a bool and True for this plate", lambda: run_valid is True)

**Your answer - judgement calls.** Where did the rules force a choice that could change the result (e.g. negative net values, `ddof=1`, treating `"OOR >"` as missing, only 2 blank wells...)? Pick **two** and explain what you'd ask the lab scientist before shipping this to production.

> _(write here)_

---
# Part 4 - Visualization

Use matplotlib (or seaborn/plotly if you prefer - add it to `requirements.txt`). Charts must be readable by a lab technician: titles, axis labels, units, legend/colorbar.

### 4.1 Plate heatmap
Write `plot_plate(results, analyte, ax=None)` that draws the **8 x 12 plate** (rows `A`-`H` top to bottom, columns 1-12 left to right) coloured by `net_mfi` on a **log colour scale** (negatives clipped to 1 *for display only*). Mark the well types (e.g. letters `B`/`S`/`N`/`P` for blank/standard/neg/pos) and mark `INVALID` wells so they are not mistaken for low signal. Return the `Axes`.

In [ ]:
def plot_plate(results: pd.DataFrame, analyte: str, ax=None):
    raise NotImplementedError


ax = plot_plate(results, "AN_A")
plt.show()

In [ ]:
check("returns an Axes with an image", lambda: hasattr(ax, "images") and len(ax.images) == 1)
check("grid is 8 x 12", lambda: ax.images[0].get_array().shape == (8, 12))
check("axes have a title and a colorbar", lambda: ax.get_title() != "" and len(ax.figure.axes) >= 2)

### 4.2 Standard curve
The six `standard` wells have known concentrations (`std_conc`, pg/mL). Write `plot_standard_curve(results, layout, analyte)`: scatter of `std_conc` vs `net_mfi` on **log-log** axes with a straight-line fit in log10 space (`np.polyfit`). Return `(slope, intercept, r2)`.

In [ ]:
def plot_standard_curve(results, layout, analyte):
    raise NotImplementedError


slope, intercept, r2 = plot_standard_curve(results, layout, "AN_A")
plt.show()
print(f"slope={slope:.2f} intercept={intercept:.2f} r2={r2:.3f}")

In [ ]:
check("returns three numbers", lambda: all(np.isfinite([slope, intercept, r2])))
check("slope is positive and r2 in (0.8, 1]", lambda: slope > 0 and 0.8 < r2 <= 1)

### 4.3 Patients vs cutoff
One figure, 4 panels (one per analyte): every **valid patient** `net_mfi` as a dot (log y-axis, jitter on x), coloured by call (POS/NEG), with the cutoff drawn as a horizontal dashed line.

In [ ]:
# your plotting code here

**Your answer - reading the plots.** In 3-4 sentences, what do these three charts tell the lab (is the run trustworthy? what would you check next?). Also: the standard curve is not a perfect straight line in log-log - why, and which curve model do labs normally use instead?

> _(write here)_

---
# Part 5 - FastAPI backend (+ optional front-end)

Work in the `app/` folder (not in the notebook):

| file | state | your job |
|---|---|---|
| `app/schemas.py` | provided | read it; finish the `TODO` (duplicate wells / sample ids -> HTTP 422) |
| `app/analysis.py` | stubs | move your **Part 3** logic here as `clean_readings()` and `analyze()` (same rules) |
| `app/main.py` | skeleton | storage class + 4 endpoints |
| `tests/test_api.py` | provided | must pass: `pytest -q`. Add **2-3 tests of your own** |

### The API contract
| Method & path | Body | Success | Errors |
|---|---|---|---|
| `GET /health` | - | `200 {"status":"ok"}` *(given)* | |
| `POST /runs` | `RunCreate` (name + samples) | `201 RunOut` | `422` bad well / duplicates / empty |
| `GET /runs/{run_id}` | - | `200 RunOut` | `404` |
| `POST /runs/{run_id}/readings` | `list[ReadingIn]` (raw rows) | `200 {"added": n}` (n = rows received) | `404` |
| `GET /runs/{run_id}/results` | - | `200 ResultsOut` (one call per `(well, analyte)`, cutoffs, `run_valid`) | `404`; `409` if no readings uploaded yet |

Design rules: keep **HTTP in `main.py`, storage in `RunRepository`, science in `analysis.py`**; endpoints get the repository through `Depends(get_repo)` (don't rename it - tests override it); runs are kept in memory.

Run it with `uvicorn app.main:app --reload` and try it at http://127.0.0.1:8000/docs.

### Bonus (optional)
`static/index.html` is served at `/ui`. Make it a tiny page that asks for a run id, calls `/runs/{id}/results`, and draws the 8x12 plate coloured by call (POS/NEG/INVALID) for a chosen analyte - plain HTML + `fetch`, no framework required. (Using a framework is fine, but be ready to explain it.)

In [ ]:
# Smoke test straight from the notebook (works once your endpoints exist)
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)
print(client.get("/health").json())

**Your answer - design questions** (2-3 sentences each)

> 1. Why use `Depends(get_repo)` instead of a global variable inside the endpoint functions? _(write here)_
>
> 2. Why keep the analysis in `analysis.py` instead of inside the endpoint? What did that make easier? _(write here)_
>
> 3. Why `409` (and not `404` or `400`) when results are requested before readings exist? _(write here)_
>
> 4. If we swapped the in-memory dict for SQLite tomorrow, which files change and which do not? _(write here)_

---
# Part 6 - System design  *(in `DESIGN_ANSWER.md`)*
Answer the scenario in **`DESIGN_ANSWER.md`** (about one page; bullets and a simple ASCII/Mermaid diagram are fine). You will present it in the interview.

# Part 7 - Wrap-up
* Fill `AI_LOG.md`.
* *Restart kernel & Run All* - the notebook should run top to bottom without errors.
* `pytest -q` is green.
* Commit and push; send us the link. **Do not** include data that isn't in this repo.